In [1]:
# Importing modules
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, auc
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Reading lipid df
df = pd.read_csv("final_data.tsv")

# Discovery data
train_df = df.loc[df["Datatype"]=="Training"]
X_train = train_df.drop(["Class", "Samples", "Datatype"], axis=1)
y_train = train_df["Class"]

# Validation data
val_df = df.loc[df["Datatype"]=="Validation"]
X_val = val_df.drop(["Class", "Samples", "Datatype"], axis=1)
y_val = val_df["Class"]

In [3]:
# Selected features
f1 = list(pd.read_excel("1_NAFLD_Analysis_Top_1_Features_List.xlsx", sheet_name="1_NF_NH_Topmost_2_List")["Lipid"])
#f2 = list(pd.read_excel("1_NAFLD_Analysis_Top_1_Features_List.xlsx", sheet_name="2_NS_NH_Topmost_2_List")["Lipid"])
#f3 = list(pd.read_excel("1_NAFLD_Analysis_Top_1_Features_List.xlsx", sheet_name="3_NS_NF_Topmost_2_List")["Lipid"])
selected_features = list(set(f1))

# Subsetting feature matrix with selected features
X_train = pd.DataFrame(X_train, columns=selected_features)
X_test = pd.DataFrame(X_val, columns=selected_features)

In [4]:
# Label encoding
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)
y_test = label_encoder.transform(y_val)

In [5]:
# Scaling the data
sds = StandardScaler()
X_train_orig, y_train_orig = X_train, y_train
X_train = pd.DataFrame(sds.fit_transform(X_train), columns=X_train.columns)
X_train_orig = pd.DataFrame(sds.fit_transform(X_train_orig), columns=X_train_orig.columns)
X_test = pd.DataFrame(sds.transform(X_test), columns=X_test.columns)

In [6]:
# Models
models = {
    'LogisticRegression': LogisticRegression(random_state=42),
    'RandomForestClassifier': RandomForestClassifier(random_state=42),
    'GradientBoostingClassifier': GradientBoostingClassifier(random_state=42),
    'XGBClassifier':XGBClassifier(random_state=42),
    'SVM': SVC(random_state=42)
}

# Parameters
params = {
    'LogisticRegression': {
        'penalty': ['l1', 'l2'],
        'C': [0.1, 1],
        'solver': ['liblinear'],
        'max_iter': [100, 200]
    },
    'RandomForestClassifier': {
        'n_estimators': [20, 50],
        'max_depth': [3, 5, 10],
        'min_samples_split': [5, 10],
        'min_samples_leaf': [2, 5],
        'max_features': ['sqrt'],
        'bootstrap': [True]
    },
    'GradientBoostingClassifier': {
        'n_estimators': [50],
        'learning_rate': [0.05, 0.1],
        'max_depth': [2, 3],
        'min_samples_split': [5, 10],
        'min_samples_leaf': [2, 5],
        'max_features': ['sqrt']
    },
    'XGBClassifier': {
        'n_estimators': [50],
        'learning_rate': [0.05, 0.1],
        'max_depth': [2, 3],
        'min_child_weight': [1, 3],
        'subsample': [0.7],
        'colsample_bytree': [0.7]
    },
    'SVM': {
        'C': [0.1, 1],
        'kernel': ['linear'],
        'gamma': ['scale'],
        'class_weight': [None],
        'probability': [True]
    },
}


best_models = {}

# Model selection and hyperparameter tuning
for name, model in models.items():
    
    # Training using Grid Search with 5 fold CV
    model = GridSearchCV(model, params[name], cv=5, scoring='roc_auc')
    model.fit(X_train.to_numpy(), y_train)
    
    # Model
    model = model.best_estimator_
    best_models[name] = model

In [7]:
names = []
accuracy = []
precision = []
recall = []
f1score = []

for m in models.keys():
    model = best_models[m]
    y_pred = model.predict(X_train_orig.to_numpy())
    names.append(m)
    accuracy.append(round(accuracy_score(y_train_orig, y_pred), 2))
    precision.append(round(precision_score(y_train_orig, y_pred, average='weighted'), 2))
    recall.append(round(recall_score(y_train_orig, y_pred, average='weighted'),2))
    f1score.append(round(f1_score(y_train_orig, y_pred, average='weighted'),2))
    
# Table showing models performance on Validation Set
print("Training")
pd.DataFrame({"Model":names, "Accuracy":accuracy, "Precision":precision, "Recall":recall, "F1-Score":f1score})

Training


,Model,Accuracy,Precision,Recall,F1-Score
0,LogisticRegression,0.90,0.91,0.90,0.90
1,RandomForestClassifier,0.97,0.97,0.97,0.97
2,GradientBoostingClassifier,1.00,1.00,1.00,1.00
3,XGBClassifier,1.00,1.00,1.00,1.00
4,SVM,0.97,0.97,0.97,0.97


In [8]:
names = []
accuracy = []
precision = []
recall = []
f1score = []

for m in models.keys():
    model = best_models[m]
    y_pred = model.predict(X_test.to_numpy())
    names.append(m)
    accuracy.append(round(accuracy_score(y_test, y_pred), 2))
    precision.append(round(precision_score(y_test, y_pred, average='weighted'), 2))
    recall.append(round(recall_score(y_test, y_pred, average='weighted'),2))
    f1score.append(round(f1_score(y_test, y_pred, average='weighted'),2))
    
# Table showing models performance on Validation Set
print("Validation")
pd.DataFrame({"Model":names, "Accuracy":accuracy, "Precision":precision, "Recall":recall, "F1-Score":f1score})

Validation


,Model,Accuracy,Precision,Recall,F1-Score
0,LogisticRegression,0.83,0.83,0.83,0.83
1,RandomForestClassifier,0.93,0.94,0.93,0.93
2,GradientBoostingClassifier,0.90,0.92,0.90,0.90
3,XGBClassifier,0.90,0.92,0.90,0.90
4,SVM,0.93,0.93,0.93,0.93
